
### 详细总结
本教程是PyTorch的**Softmax分类器**专项讲解，从二分类模型回顾出发，针对多分类任务的需求讲解核心原理，并基于MNIST数据集完成完整的分类器实现，最后给出实践拓展方向，以下是分模块详细总结：

#### 一、分类任务模型的迭代：从Sigmoid到Softmax
1. 二分类回顾：以**糖尿病数据集**为基准，使用线性层+Sigmoid层实现二分类，输出单个概率值即可满足需求。
2. 多分类痛点：MNIST数据集包含**10个数字标签**，若用Sigmoid输出10个值，结果无竞争关系，无法形成符合概率定义的预测分布。
3. 核心解决方案：引入**Softmax层**，将最后一层线性输出转化为概率分布，满足两个关键条件：$P(y=i) \geq 0$、$\sum_{i=0}^{9} P(y=i)=1$。

#### 二、Softmax层与交叉熵损失核心原理
1. **Softmax层计算逻辑**
   设最后一层线性输出$Z^l \in \mathbb{R}^K$，则类别$i$的概率为：$P(y=i)=\frac{e^{z_{i}}}{\sum_{j=0}^{K-1} e^{z_{j}}}$；
   教程以$z=[0.2,0.1,-0.1]$为例，经指数、求和、归一化后得到概率$[0.38,0.34,0.28]$，直观展示计算过程。
2. **交叉熵损失（Cross Entropy Loss）**
   - 核心公式：$Loss(\hat{Y}, Y)=-Y log \hat{Y}$，其中$Y$为真实标签的**one-hot编码**，$\hat{Y}$为Softmax输出的概率分布；
   - 仅对真实标签对应的预测概率计算损失，非真实标签项因$Y=0$无贡献。
3. **PyTorch中损失函数的对应关系**
   - **CrossEntropyLoss**：直接集成Softmax+Log+NLLLoss，输入为线性层原始输出，无需手动加Softmax层，是多分类任务的首选；
   - 等价关系：$\boldsymbol{CrossEntropyLoss = LogSoftmax + NLLLoss}$（负对数似然损失）。
4. **批量计算**：支持mini-batch训练，教程以**batch_size=3**为例，展示了批量交叉熵损失的计算过程。

#### 三、MNIST数据集多分类器的PyTorch完整实现
MNIST是手写数字数据集，核心特征为**28×28像素的灰度图**，展平后为784维输入，共10个分类标签，实现流程遵循PyTorch经典的**数据-模型-损失-训练**四步走，关键参数与操作如下：
| 实现环节 | 核心操作/参数 | 关键说明 |
|----------|---------------|----------|
| 环境准备 | 导入包：torch、torchvision.transforms/datasets、DataLoader、torch.nn.functional、optim | 分别用于张量操作、数据预处理/加载、网络激活、优化器构建 |
| 数据预处理 | transforms.Compose([ToTensor(), Normalize((0.1307,), (0.3081,))]) | ToTensor将PIL图像(0-255)转为张量(0-1)；Normalize按均值/标准差标准化 |
| 数据加载 | train/test_dataset = datasets.MNIST(download=True)；DataLoader(batch_size=64, shuffle=训练集True/测试集False) | 训练集打乱数据，测试集不打乱，批量大小设为64 |
| 网络设计 | 五层线性层：784→512→256→128→64→10，每层后接**ReLU激活函数** | 输入层将28×28张量展平为784维；输出层为10维，对应10个分类，无激活（交给CrossEntropyLoss处理） |
| 损失与优化 | 损失：nn.CrossEntropyLoss()；优化器：optim.SGD(lr=0.01, momentum=0.5) | SGD优化器加入动量0.5加速训练，学习率设为0.01 |
| 训练流程 | 定义train(epoch)函数，循环执行：梯度清零→前向传播→损失计算→反向传播→参数更新 | 每300个batch统计一次平均损失，训练轮次设为10 |
| 测试流程 | 定义test()函数，使用**torch.no_grad()** 禁用梯度计算 | 用torch.max(outputs.data, dim=1)获取预测类别，计算测试集准确率，训练每轮后测试一次 |

#### 四、训练效果与实践拓展
1. 训练效果：随训练轮次增加，损失持续下降，测试集准确率逐步提升，最终可达**97%以上**；
2. 核心工具复用：全程使用PyTorch核心组件——Dataset/DataLoader实现数据加载、nn.Module继承式构建模型、optim实现参数优化；
3. 实战练习：布置**Otto Group产品分类挑战赛**作为拓展，数据集包含94个特征、多个分类标签，需复用Softmax分类器思路实现。

#### 五、基础知识点回顾与强调
1. 数据加载的关键：Dataset需实现__init__、__getitem__、__len__方法，DataLoader实现批量加载与打乱；
2. 模型构建的规范：继承nn.Module，在__init__定义层，在forward定义前向传播逻辑；
3. 梯度计算的注意：测试阶段必须禁用梯度，避免不必要的计算开销。

---

### 关键问题
#### 问题1：多分类任务中为什么不用Sigmoid层而用Softmax层？（侧重原理区别）
答案：Sigmoid层的输出是多个独立的概率值，彼此无竞争关系，无法保证所有输出的和为1，不符合概率分布的定义；而Softmax层通过指数归一化处理，将线性层的输出转化为**非负且求和为1的概率分布**，能体现不同类别间的竞争关系，更适配多分类任务中“样本属于唯一类别”的预测需求。

#### 问题2：PyTorch中的CrossEntropyLoss为什么可以直接接收线性层的输出，无需手动添加Softmax层？（侧重PyTorch API使用）
答案：因为PyTorch的CrossEntropyLoss是**集成式损失函数**，其内部已经实现了**LogSoftmax**和**NLLLoss（负对数似然损失）** 的组合逻辑，会自动将线性层的原始输出转化为概率分布后计算损失；如果手动添加Softmax层，会导致重复计算，不仅增加开销，还可能因数值精度问题影响训练效果。

#### 问题3：基于MNIST数据集实现Softmax分类器的核心流程与关键参数是什么？（侧重工程实现）
答案：核心流程为**数据预处理与加载→模型设计→损失与优化器构建→训练与测试**；关键参数包括：①数据预处理：Normalize的均值0.1307、标准差0.3081；②数据加载：batch_size=64，训练集shuffle=True；③网络结构：784→512→256→128→64→10的五层线性层，搭配ReLU激活；④训练配置：SGD优化器lr=0.01、momentum=0.5，训练轮次10，使用CrossEntropyLoss作为损失函数；⑤测试关键：torch.no_grad()禁用梯度，torch.max(dim=1)获取预测类别。

In [ ]:
# Softmax Classifier
# 1. 导入必要的库
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import torch.nn.functional as F
import torch.optim as optim


    
# 2. 准备数据
bitch_size = 64
transform  = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])
train_dataset = datasets.MNIST(root='./data/mnist/', train=True, download=True, transform=transform)
train_loader = DataLoader(dataset=train_dataset, batch_size=bitch_size, shuffle=True)
test_dataset = datasets.MNIST(root='./data/mnist/', train=False, download=True, transform=transform)
test_loader = DataLoader(dataset=test_dataset, batch_size=bitch_size, shuffle=False)
# 3. 定义模型
class Net(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.l1 = torch.nn.Linear(784, 512)
        self.l2 = torch.nn.Linear(512, 256)
        self.l3 = torch.nn.Linear(256, 128)
        self.l4 = torch.nn.Linear(128, 64)
        self.l5 = torch.nn.Linear(64, 10)
    def forward(self, x):
        x = x.view(-1,784) #这一批有 N 张 28×28 像素的单通道灰度图片，每张图片被展开成一个 784 维的向量
        x = F.relu(self.l1(x))
        x = F.relu(self.l2(x))
        x = F.relu(self.l3(x))
        x = F.relu(self.l4(x))
        return self.l5(x)

# 4. 构造损失函数和优化器
model = Net()
criterion = torch.nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.5)
# 5. 训练模型
def train(epoch):
    running_loss = 0.0
    for batch_idx,data in enumerate(train_loader):
        inputs,target = data
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, target)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        if batch_idx % 300 == 299:
            print(f"[{epoch+1}, {batch_idx+1}] loss: {running_loss/300:.3f}")
            running_loss = 0.0

# 6. 评估模型
def test():
    correct = 0
    total = 0
    with torch.no_grad():
        for data in test_loader:
            images, labels = data
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    print(f"Accuracy on test set: {100 * correct / total:.2f}%")


In [4]:
if __name__ == "__main__":
    for epoch in range(10):
        train(epoch)
        test()

[5, 300] loss: 0.103
[5, 600] loss: 0.098
[5, 900] loss: 0.097
Accuracy on test set: 96.76%
[6, 300] loss: 0.076
[6, 600] loss: 0.075
[6, 900] loss: 0.081
Accuracy on test set: 96.68%
[7, 300] loss: 0.064
[7, 600] loss: 0.060
[7, 900] loss: 0.060
Accuracy on test set: 96.95%


KeyboardInterrupt: 

Try to implement a classifier for:
• Otto Group Product Classification Challenge
• Dataset: https://www.kaggle.com/c/otto-group-product-classification-
challenge/data

## 🛒 Otto Group 产品分类挑战赛项目介绍

### 1. 项目背景
Otto Group 是全球最大的电商与物流集团之一，业务覆盖20多个国家（如德国 Otto.de、美国 Crate & Barrel、法国 3 Suisses），每天销售数百万件商品，还在不断上架新品。

由于全球基础设施差异，**同款商品在不同地区会被打上不同分类标签**，这严重影响了产品分析的准确性。因此，Otto Group 希望通过机器学习，实现更精准的商品自动分类，从而挖掘更多产品洞察。

---

### 2. 任务目标
这是一个典型的**结构化数据多分类任务**：
- **数据规模**：提供超过 **20万件商品** 的数据，每件商品包含 **93个匿名特征**（feat_1 ~ feat_93）。
- **核心目标**：构建预测模型，将商品精准划分到 **9个主类别**（Class_1 ~ Class_9）中。
- **额外价值**：竞赛获胜模型将被**开源**，供社区学习与落地。

---

### 3. 数据构成
从 Data 页面可知，数据集分为三个核心文件：
| 文件 | 用途 |
|------|------|
| `trainData.csv` | 训练集，包含 `id`、93个特征 `feat_*` 和目标标签 `target` |
| `testData.csv` | 测试集，仅包含 `id` 和93个特征，无标签，用于最终提交 |
| `sampleSubmission.csv` | 提交格式示例，展示了正确的 CSV 结构 |

---

### 4. 评价指标
竞赛使用 **多类别对数损失（Multi-class Logarithmic Loss, Logloss）** 作为唯一评价标准：
\[
\text{logloss} = -\frac{1}{N} \sum_{i=1}^{N} \sum_{j=1}^{M} y_{ij} \log(p_{ij})
\]
- **$N$**：测试集商品总数
- **$M$**：类别数（固定为9）
- **$y_{ij}$**：真实标签（商品i属于类别j则为1，否则为0）
- **$p_{ij}$**：模型预测商品i属于类别j的概率

**核心特点**：Logloss 不仅惩罚分类错误，还会惩罚“错误的高置信度”，要求模型输出**校准良好的概率分布**，而非单纯的类别预测。

---

### 5. 提交要求
你需要提交一个 CSV 文件，格式严格如下：
- 表头必须为：`id,Class_1,Class_2,Class_3,Class_4,Class_5,Class_6,Class_7,Class_8,Class_9`
- 每行对应一个测试集商品，`id` 为商品唯一标识，`Class_*` 列填写该商品属于对应类别的概率
- 概率无需手动归一化（系统会自动处理），但会被截断到 `[1e-15, 1-1e-15]` 区间以避免数值错误

---


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset,DataLoader
import numpy as np
from sklearn.preprocessing import StandardScaler  # 引入数据归一化工具
class ottoDataset(Dataset):
    def __init__(self, inpath) -> None:
        # 1. 读取CSV文件（跳过表头，分开特征和标签）
        try:
            data = np.genfromtxt(
                inpath, 
                delimiter=',', 
                skip_header=1,  # 跳过第一行表头
                dtype=str       # 先按字符串读取，避免标签解析错误
            )
        except FileNotFoundError:
            raise FileNotFoundError(f"文件不存在：{inpath}，请检查路径是否正确！")
        
        # 2. 分离特征和标签
        features = data[:, 1:-1].astype(np.float32)
        
        # 特征归一化（解决梯度爆炸）
        scaler = StandardScaler()
        features = scaler.fit_transform(features)
        
        self.x_data = torch.from_numpy(features)
        self.y_data = self._encode_labels(data[:, -1])
        self.length = len(self.x_data)

    def _encode_labels(self, labels):
        """将Class_*字符串标签编码为0-8的数字"""
        label_map = {f'Class_{i}': i-1 for i in range(1, 10)}
        encoded = np.array([label_map[label] for label in labels])
        return torch.from_numpy(encoded).long()

    def __getitem__(self, index):
        return self.x_data[index], self.y_data[index]

    def __len__(self):
        return self.length
        
train_dataset = ottoDataset('data/otto-group-product-classification-challenge/train.csv')
train_loader = DataLoader(dataset=train_dataset,batch_size=300,shuffle=True,num_workers=0)
# test_dataset = ottoDataset('data/otto-group-product-classification-challenge/test.csv')
# test_loader = DataLoader(dataset=test_dataset,batch_size=500,shuffle=False,num_workers=0)

class OttoModule(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.l1 = torch.nn.Linear(93, 256)
        self.l2 = torch.nn.Linear(256, 128)
        self.l3 = torch.nn.Linear(128, 64)
        self.l4 = torch.nn.Linear(64, 32)
        self.l5 = torch.nn.Linear(32, 9)
    def forward(self, x):
        x = F.relu(self.l1(x))
        x = F.relu(self.l2(x))
        x = F.relu(self.l3(x))
        x = F.relu(self.l4(x))
        return self.l5(x)
model = OttoModule()
criterion = torch.nn.CrossEntropyLoss(reduction='sum')
optimizer = torch.optim.SGD(model.parameters(),lr=0.001,momentum=0.9)

def train(epoch):
    running_loss = 0.0
    model.train()  # 显式设置训练模式（规范）
    for batch_idx, (inputs, target) in enumerate(train_loader):
        optimizer.zero_grad()
        output = model(inputs)
        loss = criterion(output, target)
        loss.backward()
         # 梯度裁剪：防止极端情况梯度爆炸（可选但推荐）
        # torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        running_loss += loss.item()
        if batch_idx % 100 == 99:
            print(f"[{epoch+1}, {batch_idx+1}] loss: {running_loss/100:.3f}")
            running_loss = 0.0
def test():
    correct = 0
    total = 0
    model.eval()  # 显式设置评估模式（规范）
    with torch.no_grad():
        for data in train_loader:
            inputs, target = data
            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)
            total += target.size(0)
            correct += (predicted == target).sum().item()
    print(f'Accuracy on train set: {100 * correct / total:.2f} %')
            

if __name__ == '__main__':
    for epoch in range(50):
        train(epoch)
        test()




In [9]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
from sklearn.preprocessing import StandardScaler

class ottoDataset(Dataset):
    def __init__(self, inpath) -> None:
        try:
            data = np.genfromtxt(
                inpath, 
                delimiter=',', 
                skip_header=1,
                dtype=str
            )
        except FileNotFoundError:
            raise FileNotFoundError(f"文件不存在：{inpath}，请检查路径是否正确！")
        
        features = data[:, 1:-1].astype(np.float32)
        scaler = StandardScaler()
        features = scaler.fit_transform(features)
        
        self.x_data = torch.from_numpy(features)
        self.y_data = self._encode_labels(data[:, -1])
        self.length = len(self.x_data)

    def _encode_labels(self, labels):
        label_map = {f'Class_{i}': i-1 for i in range(1, 10)}
        encoded = np.array([label_map[label] for label in labels])
        return torch.from_numpy(encoded).long()

    def __getitem__(self, index):
        return self.x_data[index], self.y_data[index]

    def __len__(self):
        return self.length
        
train_dataset = ottoDataset('data/otto-group-product-classification-challenge/train.csv')
train_loader = DataLoader(dataset=train_dataset,batch_size=300,shuffle=True,num_workers=0)

# 改进模型：增加Dropout正则化 + BatchNorm稳定训练
class OttoModule(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.l1 = torch.nn.Linear(93, 256)
        self.bn1 = nn.BatchNorm1d(256)  # 批量归一化，稳定梯度
        self.l2 = torch.nn.Linear(256, 128)
        self.bn2 = nn.BatchNorm1d(128)
        self.l3 = torch.nn.Linear(128, 64)
        self.bn3 = nn.BatchNorm1d(64)
        self.l4 = torch.nn.Linear(64, 32)
        self.bn4 = nn.BatchNorm1d(32)
        self.l5 = torch.nn.Linear(32, 9)
        self.dropout = nn.Dropout(0.2)  # 正则化，防止过拟合

    def forward(self, x):
        x = F.relu(self.bn1(self.l1(x)))
        x = self.dropout(x)
        x = F.relu(self.bn2(self.l2(x)))
        x = self.dropout(x)
        x = F.relu(self.bn3(self.l3(x)))
        x = self.dropout(x)
        x = F.relu(self.bn4(self.l4(x)))
        return self.l5(x)

model = OttoModule()
criterion = torch.nn.CrossEntropyLoss(reduction='sum')
# 改进1：改用Adam优化器（比SGD更稳定，不易发散）
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)
# 改进2：学习率衰减（训练后期逐步降低学习率）
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=3
)

def train(epoch):
    running_loss = 0.0
    model.train()
    epoch_loss = 0.0  # 记录整个epoch的损失，用于学习率调整
    for batch_idx, (inputs, target) in enumerate(train_loader):
        optimizer.zero_grad()
        output = model(inputs)
        loss = criterion(output, target)
        loss.backward()
        # 改进3：恢复梯度裁剪，强制限制梯度范围
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        running_loss += loss.item()
        epoch_loss += loss.item()
        
        if batch_idx % 100 == 99:
            print(f"[{epoch+1}, {batch_idx+1}] loss: {running_loss/100:.3f}")
            running_loss = 0.0
    # 根据epoch损失调整学习率
    scheduler.step(epoch_loss / len(train_loader))

def test():
    correct = 0
    total = 0
    model.eval()
    with torch.no_grad():
        for data in train_loader:
            inputs, target = data
            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)
            total += target.size(0)
            correct += (predicted == target).sum().item()
    accuracy = 100 * correct / total
    print(f'Accuracy on train set: {accuracy:.2f} %')
    return accuracy  # 返回准确率，便于监控

# 主训练逻辑：增加早停机制，防止过拟合
if __name__ == '__main__':
    best_accuracy = 0.0
    patience = 5  # 5个epoch没提升就停止
    no_improve = 0
    
    for epoch in range(30):  # 训练30个epoch
        train(epoch)
        current_acc = test()
        
        # 早停逻辑：防止过拟合和参数崩坏
        if current_acc > best_accuracy:
            best_accuracy = current_acc
            no_improve = 0
            # 保存最优模型
            torch.save(model.state_dict(), 'best_otto_model.pth')
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f"早停触发！最优准确率：{best_accuracy:.2f}%")
                break
    print(f"训练结束，最优准确率：{best_accuracy:.2f}%")

[1, 100] loss: 363.676
[1, 200] loss: 221.709
Accuracy on train set: 76.46 %
[2, 100] loss: 194.711
[2, 200] loss: 187.409
Accuracy on train set: 78.96 %
[3, 100] loss: 181.790
[3, 200] loss: 177.206
Accuracy on train set: 79.50 %
[4, 100] loss: 172.085
[4, 200] loss: 174.848
Accuracy on train set: 80.43 %
[5, 100] loss: 167.561
[5, 200] loss: 169.632
Accuracy on train set: 80.86 %
[6, 100] loss: 164.415
[6, 200] loss: 165.134
Accuracy on train set: 81.09 %
[7, 100] loss: 160.296
[7, 200] loss: 163.482
Accuracy on train set: 81.84 %
[8, 100] loss: 158.329
[8, 200] loss: 158.548
Accuracy on train set: 81.73 %
[9, 100] loss: 155.763
[9, 200] loss: 156.357
Accuracy on train set: 82.13 %
[10, 100] loss: 153.003
[10, 200] loss: 155.863
Accuracy on train set: 82.40 %
[11, 100] loss: 152.095
[11, 200] loss: 154.657
Accuracy on train set: 82.64 %
[12, 100] loss: 149.516
[12, 200] loss: 152.456
Accuracy on train set: 82.66 %
[13, 100] loss: 148.074
[13, 200] loss: 148.021
Accuracy on train set: